## MMS Training
-----

This notebook tests the utilities in `src/train/mms_duration_experiment.py` for MMS adapter fine-tuning duration sweep experiments.

The workflow:
1. **Build processor once** from the full training dataset — vocabulary, tokenizer, and feature extractor are saved to disk and reused across all runs
2. **Fine-tune per duration** — load a fresh MMS model and train on progressively larger subsets (1h → 2h → 5h → 14h)
3. **Evaluate** on the held-out test set after each run to compare WER/CER across dataset sizes

> **Note:** The processor is built from the full 14h manifest so the vocabulary stays consistent across all duration experiments, making results directly comparable.



## 1.Setup

### 1.1 Python Imports

In [1]:
## Enviroment helper variable to help 
# when running the notebook in different environments (e.g. local vs colab)
ENV = "local"  # options: "local", "colab"

In [ ]:
import sys, os
from dotenv import load_dotenv
from huggingface_hub import login
from pathlib import Path

# Add project root (recommended)
if ENV == "jupyter-hub":
    PROJECT_SRC = Path("/content/drive/MyDrive/chichewa-asr/src")
else:
    PROJECT_SRC = Path().cwd().parents[1]

# Add project src to path to allow imports from src folder
sys.path.append(str(PROJECT_SRC))

# Import functions from src.train.train_whisper
from src.train.train_whisper import load_config

from src.train.mms_duration_experiment import (
    build_processor,
    load_model_and_processor,
    prepare_train_dataset,
    prepare_test_dataset,
    run_training,
    run_evaluation,
)


### 1.2. Input Folder and Other Configuration

In [3]:
# Base data directory
DIR_BASE = Path.cwd().parents[1]
DIR_DATA = DIR_BASE.joinpath('data')

# Direcotory for test data and manifest file
DIR_TEST = DIR_DATA / "test"
FILE_MANIFEST_TEST = DIR_TEST / "metadata.csv"

# Directory for dev data and manifest file
DIR_DEV = DIR_DATA / "dev"
FILE_MANIFEST_DEV = DIR_DEV / "metadata.csv"

# Directory for nested duration based data
DIR_DEV_NESTED_DURATION = DIR_DATA / "dev_nested_duration"

# Hyperparameter config file path
FILE_CONFIG = DIR_BASE / "configs" / "mms_hparams_debug.yaml"

# Outputs directory where we keep the results of the experiments
DIR_OUTPUTS = DIR_BASE / "outputs"
DIR_RESULTS = DIR_OUTPUTS / "duration-exp-mms-1b-all"
DIR_RESULTS.mkdir(parents=True, exist_ok=True)


# Model checkpoint directory (for saving model checkpoints during training)
DIR_MODELS = DIR_BASE / "models"
DIR_MODELS_ARTIFACTS = DIR_MODELS / "artifacts"


DIR_MODEL_CHECKPOINTS = DIR_MODELS / "checkpoints"
DIR_MODEL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)


In [4]:
# ==============================================
# TRAINING CONFIGURATION
# ==============================================
DEBUG = True  # Set to False for full training

### 1.3 Hugging Face Hub Log in

In [5]:
# Log into Hugging Face Hub (optional, required if you want to push the model to the hub)
if ENV == "colab":
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)

else:
    load_dotenv()
    login(token=os.getenv("HF_TOKEN"))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 2. Train Model for Single File

### 2.1 Build Processor and Load Model

In [6]:
# =========================================
# 1. LOAD HYPERPARAMETER CONFIG
# =========================================
config = load_config(FILE_CONFIG)


print(f"Config: {FILE_CONFIG.name}")
print(f"Model: {config['model']['model_name_or_path']}")


Config: mms_hparams_debug.yaml
Model: facebook/mms-1b-all


In [7]:
# ==============================================
# 2. SETUP MODEL-SPECIFIC ARTIFACT DIRECTORIES
# ==============================================
model_id = config["model"]["model_name_or_path"]
model_name = model_id.split("/")[-1]
target_lang = config["model"]["target_lang"]

# Create model-specific artifact directories
DIR_MODEL_ARTIFACT = DIR_MODELS_ARTIFACTS / model_name
DIR_PROCESSOR_ARTIFACT = DIR_MODEL_ARTIFACT / "processor"
DIR_VOCAB_ARTIFACT = DIR_MODEL_ARTIFACT / "vocab"

DIR_PROCESSOR_ARTIFACT.mkdir(parents=True, exist_ok=True)
DIR_VOCAB_ARTIFACT.mkdir(parents=True, exist_ok=True)

print(f"Model ID: {model_id}")
print(f"Processor dir: {DIR_PROCESSOR_ARTIFACT}")
print(f"Vocab dir: {DIR_VOCAB_ARTIFACT}")

Model ID: facebook/mms-1b-all
Processor dir: /Users/dmatekenya/git-repos/chichewa-asr/models/artifacts/mms-1b-all/processor
Vocab dir: /Users/dmatekenya/git-repos/chichewa-asr/models/artifacts/mms-1b-all/vocab


In [8]:
# ==============================================
# 3. BUILD PROCESSOR
# ==============================================
processor = build_processor(
    manifest_path=FILE_MANIFEST_DEV,
    audio_dir=DIR_DEV,
    model_id=model_id,
    target_lang=target_lang,
    save_dir=DIR_PROCESSOR_ARTIFACT,
)
print(f"Vocab size: {len(processor.tokenizer)}")

  Loading processor from: /Users/dmatekenya/git-repos/chichewa-asr/models/artifacts/mms-1b-all/processor
Vocab size: 54


In [9]:
# ==============================================
# 4.LOAD MODEL AND PROCESSOR
# ==============================================
model, processor = load_model_and_processor(config, 
                                            processor_dir=DIR_PROCESSOR_ARTIFACT)


  Loading processor from: /Users/dmatekenya/git-repos/chichewa-asr/models/artifacts/mms-1b-all/processor
  Loading MMS model: facebook/mms-1b-all


Loading weights: 100%|██████████| 1096/1096 [00:00<00:00, 30664.17it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-1b-all
Key            | Status   |                                                                                            
---------------+----------+--------------------------------------------------------------------------------------------
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154, 1280]) vs model:torch.Size([54, 1280])
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154]) vs model:torch.Size([54])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  Adapter loaded for lang: nya


### 2.2 Prepare Datasets

In [10]:
# ==============================================
# 1. PREPARE TEST DATASET
# ==============================================
dataset_test = prepare_test_dataset(
    manifest_path=FILE_MANIFEST_TEST,
    audio_dir=DIR_TEST,
    processor=processor,
    audio_fname_col="audio_filename",
    duration_col="duration_seconds",
)
print(f"Test set: {len(dataset_test):,} utterances")


# ==============================================
# 2. PREPARE TRAIN DATASET
# ==============================================
DURATION_LABEL = "1h"
file_manifest_1h = DIR_DEV_NESTED_DURATION / "train_1h.csv"

dataset_train = prepare_train_dataset(
    manifest_path=file_manifest_1h,
    audio_dir=DIR_DEV,
    processor=processor,
)
print(dataset_train)


  Loading test data: /Users/dmatekenya/git-repos/chichewa-asr/data/test/metadata.csv
Total duration : 1.68 hrs  (573 utterances)
  all_data   :   573 utterances  |  1.68 hrs (100.0%)


Map (num_proc=1): 100%|██████████| 573/573 [00:09<00:00, 59.13 examples/s]


Test set: 573 utterances
  Loading train data: /Users/dmatekenya/git-repos/chichewa-asr/data/dev_nested_duration/train_1h.csv
Total duration : 1.00 hrs  (278 utterances)
  train       :   244 utterances  |  0.90 hrs  (89.9%)
  validation  :    34 utterances  |  0.10 hrs  (10.1%)


Map (num_proc=1): 100%|██████████| 34/34 [00:00<00:00, 36.95 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 244
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 34
    })
})


### 2.3 Prepare Trainer and Run Training

In [11]:
hub_model_id = f"dmatekenya/mms-1b-all-chichewa-{DURATION_LABEL}"
output_dir = DIR_MODEL_CHECKPOINTS / f"mms-1b-all-chichewa-{DURATION_LABEL}"

trainer = run_training(
    model=model,
    processor=processor,
    dataset_train=dataset_train,
    run_config=config,
    hub_model_id=hub_model_id,
    output_dir=output_dir,
    debug=DEBUG,
)


  DEBUG: forcing CPU (MPS does not support CTC loss)
  Training ...


Step,Training Loss,Validation Loss,Wer
10,29.696350,28.033882,1.000000
20,15.962534,17.269329,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]


### 2.4 Run Evaluation

In [12]:
df_results = run_evaluation(model, processor, dataset_test, 
                            DURATION_LABEL, DIR_RESULTS, 
                            model_id=hub_model_id, debug=DEBUG)

[DEBUG] Running evaluation on a small sample of the test set.
  WER (corpus): 100.00%   CER (corpus): 165.78%
  Predictions saved: /Users/dmatekenya/git-repos/chichewa-asr/outputs/duration-exp-mms-1b-all/predictions_1h.csv


## 2. Train Model for Multiple Files 

In [13]:
# ================================
# BUILD DURATION_DATASETS 
# ===============================
DURATION_DATASETS = {
    f"{h}h": DIR_DEV_NESTED_DURATION / f"train_{h}h.csv"
    for h in range(1, 3)
}

# Sanity-check that all manifests exist before launching long runs
missing = [str(p) for p in DURATION_DATASETS.values() if not Path(p).exists()]
if missing:
    print("WARNING — manifest files not found:")
    for m in missing:
        print(f"  {m}")
else:
    print(f"All {len(DURATION_DATASETS)} manifest files found. Ready to run.")

All 2 manifest files found. Ready to run.


In [ ]:
# ==========================================
# EXPERIMENT SETTINGS
# ==========================================
model_id     = config["model"]["model_name_or_path"]
model_name   = model_id.split("/")[-1]
HUB_MODEL_ID = f"dmatekenya/{model_name}-chichewa"

DEBUG = True  # set False for real runs on the server

summary = []

for duration_label, manifest_path in DURATION_DATASETS.items():
    if not manifest_path.exists():
        print(f"Skipping {duration_label} — manifest not found.")
        continue

    print(f"\n{'='*60}\n  EXPERIMENT: {duration_label}\n{'='*60}")

    hub_model_id = f"{HUB_MODEL_ID}-{duration_label}"
    output_dir   = DIR_MODEL_CHECKPOINTS / f"{model_name}-chichewa-{duration_label}"

    # 1. Load fresh model (processor is shared across all runs)
    model, processor = load_model_and_processor(config, processor_dir=DIR_PROCESSOR_ARTIFACT)

    # 2. Prepare training data
    dataset_train = prepare_train_dataset(manifest_path, DIR_DEV, processor)

    # 3. Train
    train_start = time.time()
    trainer = run_training(model, processor, dataset_train, config, hub_model_id, output_dir, debug=DEBUG)
    train_minutes = (time.time() - train_start) / 60

    # 4. Push to Hub (skipped in debug)
    if not DEBUG:
        print(f"  Pushing to Hub: {hub_model_id}")
        trainer.push_to_hub()

    # 5. Evaluate on held-out test set
    df_results = run_evaluation(model, processor, dataset_test, duration_label, DIR_RESULTS, model_id=hub_model_id, debug=DEBUG)
    summary.append({
        "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "duration":      duration_label,
        "wer":           df_results["wer_avg"].iloc[0],
        "cer":           df_results["cer_avg"].iloc[0],
        "hub_model_id":  hub_model_id,
        "train_minutes": round(train_minutes, 2),
    })

    # Save rolling summary so partial results survive a crash
    pd.DataFrame(summary).to_csv(DIR_RESULTS / "duration_sweep_summary.csv", index=False)

print("\nSweep complete.")
pd.DataFrame(summary)


## Load and Prepare the Model for Training

In [ ]:
hub_model_id = f"dmatekenya/mms-1b-all-chichewa-{DURATION_LABEL}"
output_dir = DIR_MODEL_CHECKPOINTS / f"mms-1b-all-chichewa-{DURATION_LABEL}"

trainer = run_training(
    model=model,
    processor=processor,
    dataset_train=dataset_train,
    run_config=config,
    hub_model_id=hub_model_id,
    output_dir=output_dir,
)


### 1.4 Device for Training

In [ ]:
# Check if CUDA is available and set the device accordingly
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")

## 2. Define Utility Functions

## 2.MMS Training Process 

### 2.1 Prepare Data and Vocabulary 
- Clean the transcripts 
- Create Character level vocubulary list
- Save vocabulary file 

In [ ]:
# ===================================
# 1. LOAD TRAIN AND TEST DATASETS 
# ===================================
# Lets use one hour training data for quick testing
file_manifest_1h = DIR_DEV_NESTED_DURATION / "train_1h.csv"
dataset_train = load_audio_data(file_manifest_1h, audio_dir=DIR_DEV)
dataset_test = load_audio_data(FILE_MANIFEST_TEST, audio_dir=DIR_TEST,audio_fname_col="audio_filename", 
                               split_data=False, duration_col="duration_seconds")

In [ ]:
# ===================================
# 2. VIEW DATASET
# ===================================
remove_columns = ['audio', 'duration', 'audio_fname']
show_random_elements(dataset_train['train'].remove_columns(remove_columns), num_examples=10)

In [ ]:
# ===================================
# 3. PREPROCESS THE DATASET 
# ===================================
dataset_train = dataset_train.map(remove_special_characters)
dataset_test = dataset_test.map(remove_special_characters)


In [ ]:
show_random_elements(dataset_train['train'].remove_columns(remove_columns), num_examples=10)

In [ ]:
# ==================================
# 3. BUILD VOCABULARY
# ==================================
vocab_train = dataset_train['train'].map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=dataset_train['train'].column_names)
vocab_validation = dataset_train['validation'].map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=dataset_train['validation'].column_names)

# Combine the vocabularies from train and validation sets and sort them
vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_validation["vocab"][0]))
vocab_list.sort()

vocab_dict = {v: k for k, v in enumerate(vocab_list)}

print(f"Vocabulary size: {len(vocab_list)}")
print(f"Vocabulary: {vocab_dict}")

# ==================================
# 4. NORMALIZE VOCABULARY
# ==================================
# Replace space with | to avoid confusion with padding token
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

# Add special tokens to the vocabulary
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
len(vocab_dict)

# =====================================
# 5. CREATE TARGET LANGUAGE VOCAB_DICT
# AND SAVE AS JSON FILE
# =====================================
target_lang = "nya"
new_vocab_dict = {target_lang: vocab_dict}

# vocb file
vocab_file_path = DIR_MODELS / "mms-1b-chichewa" / "vocab.json"
vocab_file_path.parent.mkdir(parents=True, exist_ok=True)

with open(vocab_file_path, 'w') as vocab_file:
    json.dump(new_vocab_dict, vocab_file)


### 2.2  Load Tokenizer
We use ```Wav2Vec2CTCTokenizer```

In [ ]:
# Load the tokenizer using the vocab file
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(str(vocab_file_path.parent), unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|", target_lang=target_lang)

# Push the tokenizer to Hugging Face Hub
repo_name = "mms-1b-chichewa"
tokenizer.push_to_hub(repo_name)

### 2.3  Feature Extractor
In Wav2Vec2/XLS-R/MMS ASR models, the feature extractor prepares raw speech waveforms into the format expected by the pretrained model. Unlike traditional ASR systems that use handcrafted features such as MFCCs, these models operate directly on sampled audio signals.

The feature extractor:
- Takes raw audio arrays as input (typically mono 16 kHz waveforms)
- Outputs normalized and padded tensors (`input_values`)
- Optionally returns an `attention_mask` for batched inference/training

Sampling rate is critical because pretrained checkpoints expect audio sampled at the same rate used during pretraining (commonly 16 kHz). Using a different sampling rate changes the signal distribution and can significantly reduce performance.

Typical configuration:
- `feature_size=1` → model consumes raw waveform amplitudes directly
- `sampling_rate=16000`
- `padding_value=0.0`
- `do_normalize=True` → zero-mean/unit-variance normalization
- `return_attention_mask=True` → especially important for XLS-R/MMS batching

In [ ]:
# Feature extractor
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)

# Processor - combines the feature extractor and tokenizer
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

### 2.3 Prepare Audio Data

The `train_mms.prepare_mms_batch` function transforms raw audio recordings and transcripts into the numerical format required for training Wav2Vec2/XLS-R/MMS ASR models.

The function:
- Extracts the raw audio waveform from `batch["audio"]`
- Applies the processor/feature extractor to generate normalized `input_values`
- Computes and stores the processed audio length as `input_length`
- Converts the transcript (`sentence`) into tokenized integer `labels`

The resulting dataset example contains:
- `input_values` → processed audio input for the model
- `input_length` → length of the processed audio
- `labels` → tokenized transcript targets used during training

In [ ]:
# =========================================
# 1. SET SAMPLE RATE TO 16KHZ FOR WAV2VEC2
# =========================================
dataset_train = dataset_train.cast_column("audio", Audio(sampling_rate=16_000))
dataset_test = dataset_test.cast_column("audio", Audio(sampling_rate=16_000))

# =========================================
# 2. PREPARE AUDIO FEATURES FOR TRAINING
# =========================================
dataset_train = dataset_train.map(
    lambda batch: prepare_mms_batch(batch, processor=processor),
    remove_columns=dataset_train["train"].column_names,
)

remove_columns = [c for c in dataset_test.column_names if c != "audio_fname"]
dataset_test = dataset_test.map(
    lambda batch: prepare_mms_batch(batch, processor=processor),
    remove_columns=remove_columns,
)


### 2.3 Setup Trainer

After preprocessing the audio data, the next step is to configure the MMS training pipeline using the Hugging Face `Trainer`.

A key component of the training setup is the data collator, which prepares batches during training. We will use `train_mms.DataCollatorCTCWithPadding`, a custom collator designed for CTC-based speech recognition models such as MMS/Wav2Vec2/XLS-R.

Unlike NLP tasks, speech recognition inputs (`input_values`) are much longer than output labels (`labels`). For example, an audio sequence may contain tens of thousands of waveform values while the corresponding transcript contains only a small number of character tokens. To improve efficiency, the data collator dynamically pads each batch only to the longest sample within that batch rather than padding all samples to the maximum length in the entire dataset.

The collator:
- Separately pads audio inputs and transcript labels
- Uses the processor to correctly handle speech and text modalities
- Replaces padded label tokens with `-100` so they are ignored during loss computation

In addition to the data collator, the training pipeline also includes:
- A `compute_metrics` function for evaluating Word Error Rate (WER)
- Loading and configuring a pretrained MMS checkpoint
- Defining training hyperparameters and optimization settings

After training, the fine-tuned model will be evaluated on the held-out test set to assess transcription performance.

In [ ]:
# ===============================================
# 1. LOAD THE DATA COLLATOR FOR CTC WITH PADDING
# ===============================================
# We use a predefined data collator in train_mms.DataCollatorCTCWithPadding
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
# ===============================================
# 2. LOAD THE MODEL FOR CTC
# ===============================================
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,
)


In [ ]:
# ===============================================
# 3. CONFIGURE ADAPTER LAYERS
# ===============================================
# Re-initialize the MMS adapter layers so they can be trained specifically
# for the target language and custom vocabulary. While it is possible to
# continue training existing adapter weights via `load_adapter(...)`,
# the pretrained vocabulary and language-specific adapters may not align
# well with the current training data. Re-initializing the adapter layers
# provides a cleaner starting point for fine-tuning on the custom dataset.
model.init_adapter_layers()

# Freeze the base model parameters and only train the adapter layers
model.freeze_base_model()

adapter_weights = model._get_adapters()
for param in adapter_weights.values():
    param.requires_grad = True

In [ ]:
# ===============================================
# 4. CONFIGURE TRAINING ARGUMENTS
# ===============================================
training_args = TrainingArguments(
  output_dir=repo_name,
  train_sampling_strategy="group_by_length",
  length_column_name="input_length",
  per_device_train_batch_size=32,
  eval_strategy="steps",
  num_train_epochs=4,
  gradient_checkpointing=True,
  fp16=True,
  save_steps=200,
  eval_steps=100,
  max_steps=100,
  logging_steps=100,
  learning_rate=1e-3,
  warmup_steps=100,
  save_total_limit=2,
  push_to_hub=True,
)
# ===============================================
# 4. INITIALIZE THE TRAINER
# ===============================================
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=partial(compute_mms_corpus_metrics, processor=processor, training_mode=True),
    train_dataset=dataset_train['train'],
    eval_dataset=dataset_train['validation'],
    processing_class=processor.feature_extractor,
)


### 2.4 Run Training

In [ ]:
trainer.train()

In [ ]:
df = run_evaluation(
    model,
    processor,
    dataset_test,
    duration_label="duration_seconds",
    results_dir=DIR_RESULTS,
    debug=True)

In [ ]:
wer = df['wer']
cer = df['cer']
df_results = df['predictions']

In [ ]:
dataset_test_sample = dataset_test.shuffle(seed=42).select(range(20))
df_results = evaluate_holdout_set(model, processor, dataset_test_sample, text_column="sentence", fname_column="audio_fname")
 

In [ ]:
df_predictions = df_results["predictions"]
cer = df_results["cer"]
wer = df_results["wer"]

In [ ]:
df_results